# ACA 2 · Computación en la Nube
## Asistente documental educativo con RAG
Over Santiago Cometa Hernández · Grupo 14

Se conserva el corpus del ACA 1. Las guías son didácticas propias, no normas de CUN. Este notebook separa pruebas locales reproducibles de validación integral pendiente.

In [1]:
from pathlib import Path
import os,sys,json
if Path.cwd().name=='notebook': os.chdir('..')
sys.path.insert(0,str(Path.cwd()))
from dotenv import load_dotenv
load_dotenv('.env')
from src.rag import Retriever, answer, documents, build_catalog
print('Fragmentos:',len(documents()))
print('Catálogo SQL:',build_catalog())

Fragmentos: 26
Catálogo SQL: 26


## 1. Corpus y trazabilidad
Se reutilizan los fragmentos del ACA1 con sus identificadores y fuentes. El CSV UCI permanece separado de la orientación documental; no se toman decisiones individuales.

In [2]:
import sqlite3
with sqlite3.connect('data/catalogo.sqlite') as db:
 print(db.execute('SELECT documento,tema,COUNT(*) FROM fragmentos GROUP BY documento,tema').fetchall())

[('Guía didáctica de acompañamiento académico', 'academico', 8), ('Guía didáctica de orientación hacia bienestar e inclusión', 'bienestar', 9), ('Guía didáctica de orientación sobre apoyos económicos', 'financiero', 9)]


## 2. Validación local diagnóstica
La siguiente ejecución usa TF-IDF y extracción literal, no un LLM ni Qdrant remoto.

In [3]:
from evaluate import run
local=run()

Caso 1/7: extractivo_no_llm (1.62 ms)
Caso 2/7: extractivo_no_llm (1.2 ms)
Caso 3/7: extractivo_no_llm (1.26 ms)
Caso 4/7: abstencion_recuperacion (1.14 ms)
Caso 5/7: extractivo_no_llm (1.18 ms)
Caso 6/7: extractivo_no_llm (1.1 ms)
Caso 7/7: abstencion_recuperacion (1.23 ms)
{
  "timestamp_utc": "2026-11-15T07:10:25.500877+00:00",
  "model_id": null,
  "embedding_model": "TF-IDF",
  "versions": {
    "numpy": "2.3.5",
    "scikit-learn": "1.8.0",
    "qdrant-client": "1.20.0"
  },
  "scope": "local / extractive",
  "llm_tested": false,
  "llm_calls": 0,
  "cloud_tested": false,
  "cases": 7,
  "median_total_ms": 1.2,
  "theme_precision_at_3": 0.7777777777777778,
  "limitations": "Muestra exploratoria de 7 consultas elegidas manualmente. La coincidencia tem\u00e1tica no mide fidelidad de generaci\u00f3n ni calidad general."
}


## 2.1. Prueba generativa local real y resultado no satisfactorio
Se ejecutó Qwen2.5-0.5B-Instruct con MiniLM y Qdrant en memoria. Hubo seis llamadas reales al LLM; cinco salidas sin citas válidas fueron bloqueadas y una salida citada solo reprodujo un encabezado. No demuestra un chatbot funcional satisfactorio. La mediana de 6314 ms incluye estas salidas y no es un SLA. Qwen1.5B se intentó después: FP32 excedió memoria y BF16 falló durante inferencia en este entorno. No se inventan resultados de la alternativa.

In [4]:
baseline=json.loads(Path('results/baseline_qwen05b.json').read_text())
print({k:baseline[k] for k in ['model_id','scope','llm_calls','cloud_tested','cases','median_total_ms','theme_precision_at_3','assessment']})

{'model_id': 'Qwen/Qwen2.5-0.5B-Instruct', 'scope': 'semantic_local / hf', 'llm_calls': 6, 'cloud_tested': False, 'cases': 7, 'median_total_ms': 6314.15, 'theme_precision_at_3': 0.7777777777777778, 'assessment': 'No satisfactorio: cinco respuestas rechazadas por citas y una respuesta sin contenido útil.'}


### Reintentar integración local o cloud
Instalar requirements-local-llm.txt permite seleccionar el proveedor hf, pero esta ruta es experimental y falló en el entorno de preparación. Ruta recomendada en el computador del estudiante: Ollama y Qdrant Cloud originales, verificando cada respuesta antes de sustentar. Configurar RUN_INTEGRATION=1 y LLM_PROVIDER=ollama, o un proveedor API autorizado.

## 3. Integración cloud y generación
Activar RUN_INTEGRATION=1 solo después de configurar acceso legítimo al clúster ACA1 y modelo. No se usa una salida simulada como si fuera real.

In [5]:
if os.getenv('RUN_INTEGRATION')=='1':
 integration=run('cloud',os.getenv('LLM_PROVIDER','ollama'))
else:
 print('NO EJECUTADO: validación cloud + LLM pendiente de acceso/configuración.')

NO EJECUTADO: validación cloud + LLM pendiente de acceso/configuración.


## 4. Pruebas, análisis y límites
Revisar results/evaluation_local_extractive.json. Coincidencia temática Top-3 no equivale a fidelidad semántica. Para el RAG generativo deben revisarse afirmación por afirmación las citas, abstención fuera de corpus y latencia real. No se deben mezclar métricas locales con resultados cloud.

In [6]:
import subprocess
r=subprocess.run([sys.executable,'-m','pytest','-q'],capture_output=True,text=True)
print(r.stdout)
assert r.returncode==0

................                                                         [100%]
=============================== warnings summary ===============================
../../../opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/anyio/pytest_plugin.py:14
  /opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/python3.12/site-packages/anyio/pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
16 passed, 1 warning in 3.03s



## 5. Aplicación y sustentación
Ejecutar `streamlit run app.py`. Mostrar consulta, respuesta con fuentes, historial y panel documental. README contiene instalación y guion de demostración. Referencias: Lewis et al. (2020), https://arxiv.org/abs/2005.11401; Qdrant, https://qdrant.tech/documentation/; fuentes MEN en corpus/manifest.json.

## 6. Mejora v3 y evidencia generativa local
La vía cuantizada llama.cpp b12400 con Qwen2.5-1.5B-Instruct Q4_K_M pudo ejecutarse. Una auditoría detectó negaciones truncadas: ahora se verifica SHA256 del documento y se recuperan párrafos completos antes de generar. De cuatro casos v3, organización del estudio produjo una respuesta útil con cita; tasa de financiación reconoció que el dato no está disponible. Becas y fecha específica fueron bloqueadas por citas inválidas. Esta muestra limitada demuestra ejecución local y controles, no calidad general ni conexión Qdrant Cloud.

In [7]:
# Evidencia guardada de ejecuciones reales v3; esta celda no vuelve a llamar al LLM.
for name in ['quantized_v3_internal.json','audit_boundaries_llamacpp_prompt_v3.json']:
 evidence=json.loads(Path('results',name).read_text())
 print(name)
 for row in evidence['reports']:
  print(row.get('question'),row.get('status'),row.get('answer'))

quantized_v3_internal.json
¿Cómo organizar el estudio semanal? generado_con_citas_sintacticas [F1] Para una consulta sobre hábitos de estudio, se propone construir un horario semanal que incluya clases, trabajo independiente y descansos. El plan debe distribuir las tareas antes de las fechas de entrega y reservar tiempo para solicitar retroalimentación. El seguimiento puede registrar tareas realizadas, dudas pendientes y la siguiente acción acordada. No se recomienda interpretar la falta de tiempo como una falla moral del estudiante.
¿Qué revisar antes de solicitar una beca? salida_no_validada No dispongo de evidencia suficiente en estas guías para responder. Consulta la fuente institucional correspondiente. La salida del modelo no presentó citas válidas; no se muestra como respuesta sustentada.
audit_boundaries_llamacpp_prompt_v3.json
¿Cuál es la fecha límite de inscripción de becas CUN en noviembre de 2026? salida_no_validada No dispongo de evidencia suficiente en estas guías para re

In [8]:
if os.getenv('RUN_QUANTIZED_LOCAL')=='1':
 run('semantic_local','llamacpp')
else:
 print('Para reejecutar el modelo: configurar rutas LLAMA_CPP_BIN y GGUF_MODEL_PATH, luego RUN_QUANTIZED_LOCAL=1.')

Para reejecutar el modelo: configurar rutas LLAMA_CPP_BIN y GGUF_MODEL_PATH, luego RUN_QUANTIZED_LOCAL=1.


## Verificación posterior: Qdrant Cloud original
El reporte siguiente procede de una ejecución real externa del script `verify_cloud_rag.py`, no de volver a ejecutar todas las celdas originales del notebook. El lector necesita sus propias credenciales privadas para repetirla. Se conservan las salidas diagnósticas previas sin atribuirlas a cloud.

In [9]:
from pathlib import Path
import json
base = Path.cwd() if (Path.cwd() / 'results').exists() else Path.cwd().parent
cloud = json.loads((base / 'results/cloud_rag_acceptance.json').read_text())
print('Corpus coincidente:', cloud['preflight']['matching_points'], '/', cloud['preflight']['expected_points'])
for case in cloud['reports']:
    print(case['question'], '=>', case['status'], '| LLM:', case['llm_called'])


Corpus coincidente: 26 / 26
¿Cómo organizar el estudio semanal? => generado_con_citas_sintacticas | LLM: True
¿Qué revisar antes de solicitar una beca? => salida_no_validada | LLM: True
¿Cuál es la tasa de financiación de matrícula? => generado_con_citas_sintacticas | LLM: True
¿Cuál es la fecha límite de becas CUN en noviembre de 2026? => abstencion_recuperacion | LLM: False
